In [0]:
from pyspark.sql import functions as F, Window

S, G = "workspace.silver", "workspace.gold"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {G}")

def salvar(df, nome):
    df.write.format("delta").option("overwriteSchema", "true").mode("overwrite").saveAsTable(f"{G}.{nome}")

def com_sk(df, nome_sk, ordem):
    # Surrogate key BIGINT via row_number sobre a chave natural: sequencial e determinística
    return df.withColumn(nome_sk, F.row_number().over(Window.orderBy(*ordem)).cast("bigint"))

# ---------- dim_movies ----------
dim_movies = com_sk(spark.table(f"{S}.tb_info_filmes"), "sk_movie_id", ["id_filme"]).select(
    "sk_movie_id", F.col("id_filme").cast("string").alias("id_filme"), "titulo",
    F.col("data_lancamento").cast("date").alias("data_lancamento"),
    F.col("ano_lancamento").cast("int").alias("ano_lancamento"),
    F.col("duracao_minutos").cast("int").alias("duracao_minutos"),
    "idioma_original", "status_filme", "sinopse")
salvar(dim_movies, "dim_movies")
mov = spark.table(f"{G}.dim_movies").select("sk_movie_id", "id_filme")

# ---------- dimensões periféricas ----------
pe = spark.table(f"{S}.tb_pessoas_empresas")

dim_genres = com_sk(spark.table(f"{S}.tb_generos").select("nome_genero").distinct(),
                    "sk_genre_id", ["nome_genero"]).select("sk_genre_id", "nome_genero")
salvar(dim_genres, "dim_genres")

dim_people = com_sk(
    pe.filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
      .select(F.col("nome_entidade").alias("nome_pessoa"), F.col("tipo_entidade").alias("tipo_pessoa")).distinct(),
    "sk_person_id", ["nome_pessoa", "tipo_pessoa"]).select("sk_person_id", "nome_pessoa", "tipo_pessoa")
salvar(dim_people, "dim_people")

dim_companies = com_sk(
    pe.filter(F.col("tipo_entidade") == "Produtora").select(F.col("nome_entidade").alias("nome_produtora")).distinct(),
    "sk_company_id", ["nome_produtora"]).select("sk_company_id", "nome_produtora")
salvar(dim_companies, "dim_companies")

# ---------- dim_reviews: métrica resumida por filme ----------
rev = (spark.table(f"{S}.tb_avaliacoes_usuarios").groupBy("id_filme")
       .agg(F.count("*").cast("int").alias("qtd_avaliacoes_usuarios"),
            F.round(F.avg("nota_usuario"), 2).cast("double").alias("nota_media_usuarios"))
       .join(mov, "id_filme"))   # join garante integridade referencial com dim_movies
dim_reviews = com_sk(rev, "sk_review_id", ["sk_movie_id"]).select(
    "sk_review_id", "sk_movie_id", "qtd_avaliacoes_usuarios", "nota_media_usuarios")
salvar(dim_reviews, "dim_reviews")

# ---------- bridges (join com mov evita filmes órfãos) ----------
salvar(spark.table(f"{S}.tb_generos").join(mov, "id_filme").join(dim_genres, "nome_genero")
       .select("sk_movie_id", "sk_genre_id").distinct(), "bridge_movie_genre")

salvar(pe.filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista")).join(mov, "id_filme")
       .join(dim_people, (F.col("nome_entidade") == F.col("nome_pessoa")) & (F.col("tipo_entidade") == F.col("tipo_pessoa")))
       .select("sk_movie_id", "sk_person_id").distinct(), "bridge_movie_person")

salvar(pe.filter(F.col("tipo_entidade") == "Produtora").join(mov, "id_filme")
       .join(dim_companies, F.col("nome_entidade") == F.col("nome_produtora"))
       .select("sk_movie_id", "sk_company_id").distinct(), "bridge_movie_company")

# ---------- fato: grão = 1 linha por filme LANÇADO (silvers já são únicos por id_filme) ----------
fin = spark.table(f"{S}.tb_financeiro_filmes")
met = spark.table(f"{S}.tb_metricas_engajamento")
fato = (spark.table(f"{G}.dim_movies").filter(F.col("status_filme") == "Lançado").select("sk_movie_id", "id_filme")
        .join(fin, "id_filme", "left").join(met, "id_filme", "left")
        .select("sk_movie_id",
                *[F.col(c).cast("decimal(18,2)").alias(c) for c in
                  ["orcamento_usd", "receita_usd", "lucro_usd", "orcamento_brl", "receita_brl", "lucro_brl"]],
                F.col("popularidade").cast("double").alias("popularidade"),
                F.col("nota_media_tmdb").cast("double").alias("nota_media_tmdb"),
                F.col("qtd_votos_tmdb").cast("int").alias("qtd_votos_tmdb"),
                F.col("nota_media_imdb").cast("double").alias("nota_media_imdb"),
                F.col("qtd_votos_imdb").cast("int").alias("qtd_votos_imdb")))
salvar(fato, "fact_movies_performance")

# Conferência de grão: as duas contagens devem ser iguais
print(fato.count(), fato.select("sk_movie_id").distinct().count())

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


96261 96261


In [0]:
# 1) Receita total em R$ (a fato só tem filmes lançados)
display(spark.sql("SELECT SUM(receita_brl) AS receita_total_brl FROM workspace.gold.fact_movies_performance"))

# 2) Top 5 popularidade
display(spark.sql("""
SELECT m.titulo, f.popularidade
FROM workspace.gold.fact_movies_performance f JOIN workspace.gold.dim_movies m USING (sk_movie_id)
WHERE f.popularidade IS NOT NULL ORDER BY f.popularidade DESC LIMIT 5"""))

# 3) Filmes por gênero
display(spark.sql("""
SELECT g.nome_genero, COUNT(DISTINCT b.sk_movie_id) AS qtd_filmes
FROM workspace.gold.bridge_movie_genre b JOIN workspace.gold.dim_genres g USING (sk_genre_id)
GROUP BY g.nome_genero ORDER BY qtd_filmes DESC"""))

# 4) Top 10 receita com RANK()
display(spark.sql("""
SELECT * FROM (
  SELECT m.titulo, f.receita_usd, f.receita_brl,
         RANK() OVER (ORDER BY f.receita_usd DESC) AS posicao
  FROM workspace.gold.fact_movies_performance f JOIN workspace.gold.dim_movies m USING (sk_movie_id)
  WHERE f.receita_usd IS NOT NULL) WHERE posicao <= 10 ORDER BY posicao"""))

# 5) Ator com mais participações nos últimos 2 anos.
#    Data limite = lançamento mais recente já ocorrido (ignora datas futuras e filmes não lançados)
display(spark.sql("""
WITH lim AS (SELECT MAX(data_lancamento) AS dt_max FROM workspace.gold.dim_movies
             WHERE status_filme = 'Lançado' AND data_lancamento <= current_date())
SELECT p.nome_pessoa AS ator, COUNT(DISTINCT m.sk_movie_id) AS participacoes
FROM workspace.gold.dim_movies m CROSS JOIN lim
JOIN workspace.gold.bridge_movie_person b ON b.sk_movie_id = m.sk_movie_id
JOIN workspace.gold.dim_people p ON p.sk_person_id = b.sk_person_id
WHERE p.tipo_pessoa = 'Ator' AND m.status_filme = 'Lançado'
  AND m.data_lancamento BETWEEN add_months(lim.dt_max, -24) AND lim.dt_max
GROUP BY p.nome_pessoa ORDER BY participacoes DESC LIMIT 5"""))

# 6) Produtora com maior lucro nos últimos 5 anos (mesma data limite)
display(spark.sql("""
WITH lim AS (SELECT MAX(data_lancamento) AS dt_max FROM workspace.gold.dim_movies
             WHERE status_filme = 'Lançado' AND data_lancamento <= current_date())
SELECT c.nome_produtora, SUM(f.lucro_usd) AS lucro_usd, SUM(f.lucro_brl) AS lucro_brl
FROM workspace.gold.fact_movies_performance f
JOIN workspace.gold.dim_movies m ON m.sk_movie_id = f.sk_movie_id CROSS JOIN lim
JOIN workspace.gold.bridge_movie_company b ON b.sk_movie_id = f.sk_movie_id
JOIN workspace.gold.dim_companies c ON c.sk_company_id = b.sk_company_id
WHERE m.data_lancamento BETWEEN add_months(lim.dt_max, -60) AND lim.dt_max
GROUP BY c.nome_produtora ORDER BY lucro_usd DESC LIMIT 5"""))

receita_total_brl
810384063023.60


titulo,popularidade
blue beetle,2994.357
Gran Turismo,2680.593
The Nun II,1692.778
Meg 2: The Trench,1567.273
retribution,1547.22


nome_genero,qtd_filmes
Drama,32127
Documentary,18928
Comedy,18537
Thriller,10242
Horror,9674
Romance,7619
Action,6039
Crime,4723
Animation,4454
Tv Movie,4066


titulo,receita_usd,receita_brl,posicao
Avengers: Endgame,2800000000.00,13980120000.00,1
Avatar: The Way of Water,2320250281.00,11584777628.00,2
AVENGERS: INFINITY WAR,2052415039.00,10247503048.22,3
spider-man: no way home,1921847111.00,9595590440.51,4
The Lion King,1663075401.00,8303569169.65,5
Top Gun: Maverick,1488732821.00,7433094101.97,6
Barbie,1428545028.00,7132582470.30,7
The Super Mario Bros. Movie,1355725263.00,6769000665.63,8
Black Panther,1349926083.00,6740045939.81,9
Star Wars: The Last Jedi,1332698830.00,6654031988.31,10


ator,participacoes
Kevin Hart,64
Nathalie Emmanuel,59
Stacy Hall,59
Jay Hector,59
David Chang,59


nome_produtora,lucro_usd,lucro_brl
Universal Pictures,5772329679.00,28820664854.27
Marvel Studios,4953462823.00,24732144528.96
Columbia Pictures,3662050755.00,18284253214.65
Pascal Pictures,2701952454.00,13490578407.58
Illumination,2431353473.00,12139504755.34
